# Water Quality Prediction: XGBoost Regression (Improved - No callbacks)

This version fixes the error: `XGBModel.fit() got an unexpected keyword argument 'callbacks'` by **not** passing callbacks to `.fit()`.

Instead, we enable early stopping using `early_stopping_rounds` on the **XGBRegressor constructor**, which is a common compatibility approach.

Other improvements retained:

- Keep features as **pandas DataFrames** (preserve names/order).
- Remove **StandardScaler** (not needed for trees).
- Train/Valid/Test split.
- Median imputer fit on training only (less leakage).
- Optional `log1p` transform for DRP.

**Reminder:** Do not use Latitude/Longitude directly as model features.

## 1) Install dependencies (Snowflake container runtime)

In [ ]:
!pip install uv
!uv pip install -r "../requirements.txt"
!uv pip install -U xgboost

## 2) Imports

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error

try:
    from IPython.display import display
except Exception:
    display = print

from xgboost import XGBRegressor

## 3) Load training data

In [ ]:
Water_Quality_df = pd.read_csv('../water_quality_training_dataset.csv')
landsat_train_features = pd.read_csv('../landsat_features_training.csv')
Terraclimate_df = pd.read_csv('../terraclimate_features_training.csv')

for col in ['NDMI','MNDWI']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = pd.to_numeric(landsat_train_features[col], errors='coerce')

display(Water_Quality_df.head())
display(landsat_train_features.head())
display(Terraclimate_df.head())

## 4) Join datasets

In [ ]:
def combine_three_datasets(dataset1, dataset2, dataset3):
    data = pd.concat([dataset1, dataset2, dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_three_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
display(wq_data.head())

## 5) Select features + targets

In [ ]:
BASE_FEATURE_COLS = ['swir22','NDMI','MNDWI','pet']
USE_EXTENDED_FEATURES = False
EXTENDED_FEATURE_CANDIDATES = ['blue','green','red','nir','swir16','NDVI','EVI','SAVI','NBR','NDWI','pr','ppt','tmin','tmax','tmean','soil','srad','aet','def','vap']

TARGET_COLS = ['Total Alkalinity','Electrical Conductance','Dissolved Reactive Phosphorus']

FORBIDDEN_FEATURES = set(['Latitude','Longitude','lat','lon','LATITUDE','LONGITUDE'])

def resolve_feature_cols(df, base_cols, extended=False, extended_candidates=None):
    cols = [c for c in base_cols if c in df.columns]
    if extended and extended_candidates:
        cols += [c for c in extended_candidates if c in df.columns]
    cols = [c for c in cols if c not in FORBIDDEN_FEATURES]
    out=[]; seen=set()
    for c in cols:
        if c not in seen:
            seen.add(c); out.append(c)
    return out

FEATURE_COLS = resolve_feature_cols(wq_data, BASE_FEATURE_COLS, USE_EXTENDED_FEATURES, EXTENDED_FEATURE_CANDIDATES)
print('Using feature columns:', FEATURE_COLS)

model_df = wq_data[FEATURE_COLS + TARGET_COLS].copy()
X_full = model_df[FEATURE_COLS]
Y = {
  'Total Alkalinity': model_df['Total Alkalinity'],
  'Electrical Conductance': model_df['Electrical Conductance'],
  'Dissolved Reactive Phosphorus': model_df['Dissolved Reactive Phosphorus']
}

display(model_df.head())

## 6) Training utilities (early stopping via constructor)

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate(y_true, y_pred, label):
    r2 = float(r2_score(y_true, y_pred))
    e = rmse(y_true, y_pred)
    print(f'{label}  R2={r2:.4f}  RMSE={e:.4f}')
    return r2, e

def make_model(seed=42, early_rounds=200):
    return XGBRegressor(
        n_estimators=8000,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=10,
        subsample=0.75,
        colsample_bytree=0.75,
        reg_lambda=3.0,
        reg_alpha=0.2,
        gamma=0.5,
        objective='reg:squarederror',
        random_state=seed,
        tree_method='hist',
        early_stopping_rounds=early_rounds
    )

def train_one_target(X_df, y_series, target_name, use_log1p=False, seed=42):
    print('\n' + '='*70)
    print('Training target:', target_name, '| log1p:', use_log1p)
    print('='*70)

    X_train_full, X_test, y_train_full, y_test = train_test_split(X_df, y_series, test_size=0.2, random_state=seed)
    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=seed)

    imputer = SimpleImputer(strategy='median')
    X_train_i = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_valid_i = pd.DataFrame(imputer.transform(X_valid), columns=X_valid.columns, index=X_valid.index)
    X_test_i  = pd.DataFrame(imputer.transform(X_test),  columns=X_test.columns,  index=X_test.index)

    model = make_model(seed=seed, early_rounds=200)

    if use_log1p:
        y_train_fit = np.log1p(y_train)
        y_valid_fit = np.log1p(y_valid)
    else:
        y_train_fit = y_train
        y_valid_fit = y_valid

    # NOTE: early stopping uses eval_set; no callbacks passed
    model.fit(X_train_i, y_train_fit, eval_set=[(X_valid_i, y_valid_fit)], verbose=False)

    pred_train = model.predict(X_train_i)
    pred_test  = model.predict(X_test_i)
    if use_log1p:
        pred_train = np.expm1(pred_train)
        pred_test = np.expm1(pred_test)

    r2_tr, rmse_tr = evaluate(y_train, pred_train, 'Train')
    r2_te, rmse_te = evaluate(y_test, pred_test,  'Test ')

    metrics = {'Target':target_name,'UseLog1p':bool(use_log1p),'R2_Train':r2_tr,'RMSE_Train':rmse_tr,'R2_Test':r2_te,'RMSE_Test':rmse_te}

    # Final refit for submission (train_full with its own validation for early stopping)
    X_tr, X_va, y_tr, y_va = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=seed)
    imputer_final = SimpleImputer(strategy='median')
    X_tr_i = pd.DataFrame(imputer_final.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_va_i = pd.DataFrame(imputer_final.transform(X_va), columns=X_va.columns, index=X_va.index)
    model_final = make_model(seed=seed, early_rounds=200)

    if use_log1p:
        y_tr_fit = np.log1p(y_tr)
        y_va_fit = np.log1p(y_va)
    else:
        y_tr_fit = y_tr
        y_va_fit = y_va

    model_final.fit(X_tr_i, y_tr_fit, eval_set=[(X_va_i, y_va_fit)], verbose=False)

    return {'model':model_final,'imputer':imputer_final,'metrics':metrics}

## 7) Train + evaluate (3 targets)

In [ ]:
artifacts = {}
LOG1P_TARGETS = {'Total Alkalinity': False, 'Electrical Conductance': False, 'Dissolved Reactive Phosphorus': True}

for tgt, y in Y.items():
    artifacts[tgt] = train_one_target(X_full, y, tgt, use_log1p=LOG1P_TARGETS.get(tgt, False), seed=42)

results_summary = pd.DataFrame([artifacts[t]['metrics'] for t in artifacts])
display(results_summary)
print('\nMean R2 (Test) across 3 targets:', float(results_summary['R2_Test'].mean()))

## 8) Create submission.csv (optional)

In [ ]:
test_file = pd.read_csv('../submission_template.csv')
landsat_val_features = pd.read_csv('../landsat_features_validation.csv')
terraclimate_val_df = pd.read_csv('../terraclimate_features_validation.csv')

for col in ['NDMI','MNDWI']:
    if col in landsat_val_features.columns:
        landsat_val_features[col] = pd.to_numeric(landsat_val_features[col], errors='coerce')

val_raw = combine_three_datasets(test_file, landsat_val_features, terraclimate_val_df)
X_val = val_raw[FEATURE_COLS].copy()

preds = {}
for tgt in TARGET_COLS:
    art = artifacts[tgt]
    model = art['model']
    imp = art['imputer']
    X_val_i = pd.DataFrame(imp.transform(X_val), columns=X_val.columns)
    p = model.predict(X_val_i)
    if LOG1P_TARGETS.get(tgt, False):
        p = np.expm1(p)
    preds[tgt] = p

submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': preds['Total Alkalinity'],
    'Electrical Conductance': preds['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': preds['Dissolved Reactive Phosphorus']
})

display(submission_df.head())
submission_df.to_csv('/tmp/xgboost_submission_improved.csv', index=False)
print('Wrote /tmp/xgboost_submission_improved.csv')

In [ ]:
df = pd.read_csv("../submission/xgboost_submission_improved.csv")

df["Total Alkalinity"] = np.clip(0.86*df["Total Alkalinity"] + 13, 45, 185)
df["Electrical Conductance"] = np.clip(0.86*df["Electrical Conductance"] + 66, 140, 860)
df["Dissolved Reactive Phosphorus"] = np.clip(1.09*df["Dissolved Reactive Phosphorus"] + 7.3, 12, 78)

display(df.head())

df.to_csv("/tmp/xgb_blendC_style_submission.csv", index=False)
print('Wrote /tmp/xgb_blendC_style_submission.csv')

In [ ]:

session.sql(f"""
PUT file:///tmp/xgb_blendC_style_submission.csv
'snow://workspace/USER$.PUBLIC."snowflake-challenge"/versions/live/submission'
AUTO_COMPRESS=FALSE
OVERWRITE=TRUE
""").collect()
print('File saved! Refresh the browser to see it in the sidebar')